[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/orientation/wu_experiment/wu_experiment.ipynb)

# The Wu Experiment

This example computes the angular emission distribution of beta electrons from polarized cobalt-60 in Chien-Shiung Wu's 1957 parity experiment, alongside its optical mirror image and the physically mirrored apparatus. Polarizing cobalt-60 nuclei with the current in a solenoid sends more beta electrons out opposite to the current's circulation than along it; in an upright mirror beside the cryostat, the reflected current circulates the other way while the reflected electrons still travel downward, whereas winding the reversed solenoid in the laboratory sends electrons upward instead.

In three-dimensional geometric algebra, nuclear spin is modelled as an intrinsic angular-momentum patch `xy.dual()` in the plane of the polarizing current, and each emission direction is an intrinsic antivector. Pairing the spin patch with an emission direction via `spin & direction` produces an intrinsic scalar, so multiplying by an intrinsic weak-decay asymmetry constant in `1 + asymmetry * (spin & direction)` yields an ordinary scalar rate; reflecting the apparatus via `mirror >> spin` flips the coil's circulation while leaving the weak interaction's `asymmetry` unchanged.

In Gibbs–Heaviside vector notation, the coil current $\mathbf{J}$ defines nuclear spin as an axial pseudovector $\mathbf{I} \propto \mathbf{r} \times \mathbf{J}$ via the right-hand rule while the electron momentum $\mathbf{p}$ is a polar vector, so the textbook rate $W(\theta) = 1 + A\,\frac{\mathbf{I}\cdot\mathbf{p}}{|\mathbf{I}||\mathbf{p}|}\frac{v}{c} = 1 + A\,\frac{v}{c}\cos\theta$ hides the pseudoscalar character of $\mathbf{I}\cdot\mathbf{p}$ behind a manual sign flip $\mathbf{p} \mapsto -\mathbf{p}$, $\mathbf{I} \mapsto +\mathbf{I}$ under spatial inversion.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.util.animation import save_figure
from examples.orientation.wu_experiment import render, scenarios

np.set_printoptions(precision=3, suppress=True)

# Euclidean 3D space `x+y+z+`, with `+` extrinsic and `-` intrinsic.
ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                       # Scalar+
Direction = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                        # Vector+
Spin = ga.gatype.antibivector()                                                   # Antibivector-
Asymmetry = ga.gatype.antipseudoscalar()                                          # Antipseudoscalar-

## Beta-decay rates in the original and mirrored coils

A solenoid carrying current counterclockwise as viewed from above polarizes cobalt-60 nuclei in its horizontal plane, and beta decay departs from isotropic emission by `asymmetry * (spin & directions)`. An upright mirror containing the solenoid axis displays along each direction what the original apparatus emits along `mirror >> directions`, whereas winding the mirrored solenoid in the laboratory reflects `spin` into `mirror >> spin` under the same weak-decay `asymmetry`.

In [ ]:
n_directions = scenarios.n_directions
angles = np.linspace(0, 2 * np.pi, n_directions, endpoint=False)
asymmetry = mv(Asymmetry, [scenarios.decay_asymmetry])                            # [] Asymmetry-
spin = mv.xy.dual()                                                               # [] Spin-
mirror = mv.x                                                                     # [] Plane+
meridian_rotors = (mv.zx * (-angles / 2)).exp()                                   # [n_directions] Rotor+
directions = meridian_rotors >> mv.z.dual()                                       # [n_directions] Direction-

In [ ]:
def emission_rates(spin: Spin, directions: Direction, asymmetry: Asymmetry) -> Scalar:
    """Polarized nuclei of `spin` (`[] Spin-`) emit electrons along unit `directions`
    (`[...] Direction-`) with relative intensity `1 + asymmetry * (spin & directions)` (`[...] Scalar+`)."""
    # Pairing two intrinsic elements produces an intrinsic scalar that multiplies `asymmetry` into an ordinary rate.
    return 1 + asymmetry * (spin & directions)                                    # [...] Scalar+


def parity_cases(spin: Spin, directions: Direction, asymmetry: Asymmetry, mirror: Plane) -> tuple[Spin, Scalar]:
    """Comparing the original setup, its optical mirror image across `mirror`, and the physically
    mirrored coil under the unchanged weak interaction yields `[3] Spin-` coil spins and
    `[3, n_directions] Scalar+` angular emission rates."""
    original_rates = emission_rates(spin, directions, asymmetry)                  # [n_directions] Scalar+
    # An upright mirror displays along `directions` what the original apparatus emits along `mirror >> directions`.
    mirror_image_rates = emission_rates(spin, mirror >> directions, asymmetry)    # [n_directions] Scalar+
    # Rebuilding the mirrored coil reflects `spin` while the weak interaction retains `asymmetry`.
    mirrored_apparatus_rates = emission_rates(mirror >> spin, directions, asymmetry)  # [n_directions] Scalar+
    case_spins = stack([spin, mirror >> spin, mirror >> spin])                    # [3] Spin-
    return case_spins, stack([original_rates, mirror_image_rates, mirrored_apparatus_rates])  # [3, n_directions] Scalar+


spins, rates = parity_cases(spin, directions, asymmetry, mirror)                  # [3] Spin-, [3, n_directions] Scalar+
coil_spins = stack([spin, mirror >> spin])                                        # [2] Spin-
vertical_directions = stack([-mv.z.dual(), mv.z.dual()])                          # [2] Direction-
vertical_rates = emission_rates(coil_spins[:, None], vertical_directions, asymmetry)  # [2, 2] Scalar+
print("Downward and upward rates (original coil, then mirrored coil):")
print(vertical_rates.to_array())

figure = render.experiment_figure(spins, directions, rates)
save_figure(figure, "wu_experiment")
figure

Each panel plots the angular emission rate as radial distance from the centre in the vertical plane cutting the coil, where a cross marks wire current entering the page and a dot marks current leaving the page. From left to right, the panels show the original apparatus, its optical mirror image, and the mirrored coil built in the laboratory; the middle and right panels share the same reflected current yet emit electrons into opposite hemispheres because the weak interaction's intrinsic `asymmetry` belongs to nature rather than to the coil.

In [ ]:
# checks
# Reflecting the weak asymmetry alongside the coil would reproduce the optical mirror image, whereas physical beta decay sends electrons opposite ways.
_, mirror_image_rates, mirrored_apparatus_rates = rates                           # [n_directions] Scalar+
mirrored_law_rates = emission_rates(mirror >> spin, directions, mirror >> asymmetry)  # [n_directions] Scalar+
assert ((mirrored_law_rates - mirror_image_rates).abs() < 1e-12).all()
assert ((mirrored_apparatus_rates + mirror_image_rates - 2).abs() < 1e-12).all()
assert bool(vertical_rates[0, 0] > vertical_rates[0, 1]) and bool(vertical_rates[1, 1] > vertical_rates[1, 0])